In [8]:
import os
import json
from astropy.cosmology import FlatwCDM
import os, sys
import pymultinest
import re
import numpy as np
from astropy.io import ascii
from astropy.io import fits
from astropy.table import Table
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO
from matplotlib.ticker import AutoMinorLocator
from astropy.constants import L_sun
from matplotlib import gridspec
import astropy.units as u
from math import pi
import gc
from matplotlib.ticker import ScalarFormatter
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from astropy.table import Column
from scipy.stats import bootstrap
from getdist import plots, MCSamples
import scipy.optimize as op
from scipy import stats
from scipy.stats import norm
from astropy.time import Time
import matplotlib.dates as mdates
from joblib import Parallel, delayed
import time

In [9]:
class subset_creation:
    def __init__(self,
                 input_df,
                 condition,
                 subset_name):

        self.input_df = input_df
        self.condition = condition #Debe ser una lista de Tuplas
        self.subset_name = subset_name
        self.subset = None
        self.set_selection()

    def set_selection(self):
        tmp_dataFrame = pd.read_csv(self.input_df)
        for s in range(len(self.condition)):
            if self.condition[s][1] == '==':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]==self.condition[s][2]]
            elif self.condition[s][1] == '!=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]!=self.condition[s][2]]
            elif self.condition[s][1] == '>=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]>=self.condition[s][2]]
            elif self.condition[s][1] == '<=':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]<=self.condition[s][2]]
            elif self.condition[s][1] == '>':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]>self.condition[s][2]]
            elif self.condition[s][1] == '<':
                tmp_dataFrame = tmp_dataFrame[tmp_dataFrame[self.condition[s][0]]<self.condition[s][2]]
        self.subset = tmp_dataFrame
        self.subset.to_csv(self.subset_name,index=False)

        galaxies = list(tmp_dataFrame['galaxy'].unique())


        main_folder_name = (self.subset_name).replace('.csv','')
        if not os.path.exists(main_folder_name):
            os.makedirs(main_folder_name, exist_ok=True)
            os.makedirs(f"{main_folder_name}/Bootstrap/", exist_ok=True)
            os.makedirs(f"{main_folder_name}/Nested_sampling/", exist_ok=True)
            os.makedirs(f"{main_folder_name}/Subplots/", exist_ok=True)

            for G in range(len(galaxies)):
                os.makedirs(f"{main_folder_name}/Bootstrap/{galaxies[G]}/", exist_ok=True)
                os.makedirs(f"{main_folder_name}/Nested_sampling/{galaxies[G]}/", exist_ok=True)
                os.makedirs(f"{main_folder_name}/Subplots/{galaxies[G]}", exist_ok=True)



subset_creation(input_df = 'PLRC_general_v1.1.csv',
                condition = [('metal_correction','==',False),('random_error','>=',0),('citation','!=','Macri et al.2001')],
                subset_name = 'PLRC_nZ_eR.csv')
subset_creation(input_df = 'PLRC_general_v1.1.csv',
                condition = [('metal_correction','==',False),('total_error','>=',0),('citation','!=','Macri et al.2001')],
                subset_name = 'PLRC_nZ_eT.csv')
subset_creation(input_df = 'PLRC_general_v1.1.csv',
                condition = [('metal_correction','==',True),('random_error','>=',0),('citation','!=','Macri et al.2001')],
                subset_name = 'PLRC_Z_eR.csv')
subset_creation(input_df = 'PLRC_general_v1.1.csv',
                condition = [('metal_correction','==',True),('total_error','>=',0),('citation','!=','Macri et al.2001')],
                subset_name = 'PLRC_Z_eT.csv')
subset_creation(input_df = 'TRGB_general_v1.1.csv',
                condition = [('random_error','>=',0),('citation','!=','Macri et al.2001')],
                subset_name = 'TRGB_eR.csv')

In [16]:

class moduli_stats_calculator:
    def __init__(self,
                 dataset=None,
                 error_selection='random_error',
                 galaxy_name=None,
                 multinest_steps = 5000,
                 bootstrap_steps = 200000,
                 folder_output=None):

        self.dataset = dataset
        self.error_selection = error_selection
        self.galaxy_name = galaxy_name
        self.multinest_steps = multinest_steps
        self.bootstrap_steps = bootstrap_steps
        self.folder_output = folder_output

        self.input_data_frame = self.check_data()

        self.moduli = np.array(self.input_data_frame['modulus'])
        self.moduli_error = np.array(self.input_data_frame[error_selection])


        BOOTSTRAP = self.bootstrap_error()
        NESTED_SAMPLING = self.nestedsampling_error() 

        self.N_records = len(self.input_data_frame)
        self.mu_W = self.weighted_average()
        self.mu_B = BOOTSTRAP[0]
        self.mu_L = NESTED_SAMPLING[0]

        self.sigma_W = self.weighted_error()
        self.sigma_B = BOOTSTRAP[1]
        self.sigma_C = self.cochran_error()
        self.sigma_cl_plus = BOOTSTRAP[2][0]
        self.sigma_cl_minus = BOOTSTRAP[2][1]
        self.sigma_L = NESTED_SAMPLING[1]
        self.sigma_Lcorr = NESTED_SAMPLING[2]

        self.summary_stats()




    def check_data(self):
        """
        Funcion para checkear que toda la informacion esta bien y separar informacion por galaxia.
        """
        if self.dataset != None:
            data_frame = pd.read_csv(self.dataset)
            column_list = list(data_frame.columns)
            if not self.error_selection in column_list:
                print('Error selection not found in data frame')
            else:
                galaxy_subframe = data_frame[data_frame['galaxy']==self.galaxy_name]
                if len(galaxy_subframe) == 0:
                    print('Galaxy not found')
                else:
                    galaxy_subframe.to_csv(f'{self.dataset.replace('.csv','')}/{self.galaxy_name}_data.csv',index=False)
                    return galaxy_subframe




    def weighted_average(self):
        """
        Computo del promedio ponderado
        """
        modulus,error = self.moduli,self.moduli_error
        err_sq_inv = 1 / ((error)**2)
        num = np.sum(err_sq_inv * modulus)
        dem = np.sum(err_sq_inv)
        return num/dem

    def weighted_error(self):
        """
        Computo del error ponderado
        """
        error = self.moduli_error
        err_sq_inv = 1 / ((error)**2)
        dem = np.sum(err_sq_inv)
        return 1 / np.sqrt(dem)

    def bootstrap_error(self):
        """
        Computo del promedio, error, distribucion y percentiles mediante bootstrap
        """
        def weighted_average_x_bootstrap(modulus,error):
            modulus,error = np.array(modulus),np.array(error)
            err_sq_inv = 1 / ((error)**2)
            num = np.sum(err_sq_inv * modulus)
            dem = np.sum(err_sq_inv)
            return num/dem
        PATH = f'{self.dataset.replace('.csv','')}/Bootstrap/{self.galaxy_name}/{self.galaxy_name}.txt'
        if os.path.exists(PATH) == True:
            bootstrap_statistics = np.loadtxt(PATH)
            print(f'Reading bootstrap for {self.galaxy_name}\n')
        if os.path.exists(PATH) == False:
            print(f'Running bootstrap for {self.galaxy_name}\n')
            modulus,error = self.moduli,self.moduli_error
            size = len(modulus)
            n_bootstraps = self.bootstrap_steps  # Number of bootstrap iterations
            bootstrap_statistics = []
            for i in range(n_bootstraps):
                # Create a resample with replacement, same size as original data
                bootstrap_sample = np.random.choice(size, size=size, replace=True)
                modulus_sample = modulus[bootstrap_sample]
                error_sample = error[bootstrap_sample]
                sample_mean = weighted_average_x_bootstrap(modulus_sample,error_sample)
                bootstrap_statistics.append(sample_mean)
            bootstrap_statistics = np.array(bootstrap_statistics)
            np.savetxt(fname = PATH, X = bootstrap_statistics)

        N_dist = len(bootstrap_statistics)
        br_mean = np.mean(bootstrap_statistics)
        quad_br = (bootstrap_statistics-br_mean)**2
        lower_bound = np.percentile(bootstrap_statistics, 15.87)
        upper_bound = np.percentile(bootstrap_statistics, 84.13)

        return [br_mean,
            np.sqrt(((1)/(N_dist-1)) * np.sum(quad_br)),
            [br_mean - lower_bound, upper_bound - br_mean],
            bootstrap_statistics]




    def cochran_error(self):
        """
        Computo del error mediante el metodo de Cochran
        """
        def weighted_average_x_cochran(modulus,error):
            err_sq_inv = 1 / ((error)**2)
            num = np.sum(err_sq_inv * modulus)
            dem = np.sum(err_sq_inv)
            return num/dem
        Mu_i,Er_i = self.moduli,self.moduli_error
        n = len(Mu_i)
        w_i = 1 / ((Er_i)**2)
        Mu_w = weighted_average_x_cochran(Mu_i,Er_i)
        w_mean = np.mean(w_i)

        s1 = n / ((n-1) * (np.sum(w_i))**2)
        s2 = (w_i * Mu_i  - (w_mean * Mu_w))**2
        s3 = (w_i - w_mean) * ((w_i*Mu_i) - (w_mean*Mu_w)) 
        s4 = (w_i - w_mean)**2

        return np.sqrt(s1 * (np.sum(s2) - (2 * Mu_w * np.sum(s3)) +    (Mu_w**2 * np.sum(s4)) ) )




    def nestedsampling_error(self):
        """
        Computo del modulo de distancia y error mediante Nested Sampling
        """
        modulus,error = self.moduli,self.moduli_error

        def prior_transform(x):
            mu = 20 * x + 20
            return mu
        
        def lnlike(theta, m, merr):
            mu_pi = theta
            R = (mu_pi - m)
            W = 1.0/(merr**2)
            xsq = np.sum(R**2 * W)
            L = -0.5*xsq
            return L
        
        def loglike_for_mnest(theta):
            return lnlike(theta, modulus, error)
        
        outdir = os.path.join(self.dataset.replace('.csv',''), f'Nested_sampling/{self.galaxy_name}/')
        os.makedirs(outdir, exist_ok=True)
        prefix = os.path.join(outdir, self.galaxy_name)

        n_dims = 1
        result = pymultinest.solve(
            LogLikelihood=loglike_for_mnest,
            Prior=prior_transform,
            n_dims=n_dims,
            outputfiles_basename=prefix,
            evidence_tolerance=0.5,
            n_live_points=self.multinest_steps,
            multimodal=True,
            verbose=False,
        )

        samples = result["samples"] 
        T_sample = samples.T
        logL = np.array([loglike_for_mnest(s) for s in samples])
        chi2_map = -2.0*np.max(logL)
        N = len(modulus)   # número de datos
        k = n_dims         # parámetros del modelo
        nu = N - k
        chi2_red = chi2_map / nu
        sigma2_corr = T_sample.std() * chi2_red

        samples_x_weight  = pymultinest.analyse.Analyzer(n_params = 1, outputfiles_basename=prefix)
        mu_L = float(samples_x_weight.get_stats()['modes'][0]['mean'][0])

        return [mu_L,T_sample.std(),sigma2_corr]


    def summary_stats(self):

        PATH = os.path.join(self.dataset.replace('.csv',''), f'TRGB_eR_recal.csv')

        if not os.path.exists(PATH):
                column_x_emptyDF = ['galaxy','records','mu_W','mu_B','mu_L','sigma_W','sigma_B','sigma_C','sigma_cl+','sigma_cl-','sigma_L','sigma_Lcorr']
                Empty_DF = pd.DataFrame(columns=column_x_emptyDF)
                Empty_DF.to_csv(PATH,index=False)

        DF_comp_recal = pd.read_csv(PATH)
        if not self.galaxy_name in list(DF_comp_recal['galaxy']):
            new_row = pd.DataFrame({
                'galaxy': [self.galaxy_name],
                'records': [self.N_records],
                'mu_W': [self.mu_W],
                'mu_B': [self.mu_B],
                'mu_L': [self.mu_L],
                'sigma_W': [self.sigma_W],
                'sigma_B': [self.sigma_B],
                'sigma_C': [self.sigma_C],
                'sigma_cl+': [self.sigma_cl_plus],
                'sigma_cl-': [self.sigma_cl_minus],
                'sigma_L': [self.sigma_L],
                'sigma_Lcorr': [self.sigma_Lcorr]
            })
            DF_comp_recal = pd.concat([DF_comp_recal, new_row], ignore_index=True)
            DF_comp_recal.to_csv(PATH,index=False)


In [ ]:
moduli_stats_calculator(dataset='TRGB_eR.csv', galaxy_name='NGC5204')


Running bootstrap for NGC5204

 MultiNest Warning: no resume file found, starting from scratch
 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points = 5000
 dimensionality =    1
 *****************************************************
  analysing data from TRGB_eR/Nested_sampling/NGC5204/NGC5204.txt
 ln(ev)=  -5.2515419884865260      +/-   3.0349351970090595E-002
 Total Likelihood Evaluations:        44444
 Sampling finished. Exiting MultiNest
  analysing data from TRGB_eR/Nested_sampling/NGC5204/NGC5204.txt
 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points = 5000
 dimensionality =    1
 resuming from previous job
 *****************************************************
  analysing data from TRGB_eR/Nested_sampling/NGC5204/NGC5204.txt
 ln(ev)=  -5.2515419884865260      +/-   3.0349351970090595E-002

[28.39226952075819,
 np.float64(0.04761816660597013),
 np.float64(0.007074705315130522)]